# **Pandera: Data Validation Framework**

Pandera is a flexible and expressive API for performing data validation on dataframe-like objects. It makes data processing pipelines more readable and robust with statistically typed dataframes.

## **Table of Contents**
1. [Installation and Setup](#installation)
2. [Basic DataFrame Schema](#basic-schema)
3. [Column Validation](#column-validation)
4. [Data Type Coercion](#type-coercion)
5. [Custom Checks and Validation](#custom-checks)
6. [DataFrame Models (Class-based API)](#dataframe-models)
7. [Error Handling and Reporting](#error-handling)
8. [Advanced Features](#advanced-features)
9. [Real-world Example](#real-world-example)
10. [Best Practices](#best-practices)

## **Installation and Setup** <a id="installation"></a>

Install pandera with pandas support:

In [2]:
# Install pandera (run this in terminal if not already installed)
# pip install 'pandera[pandas]'

from typing import Optional

import numpy as np
import pandas as pd
import pandera.pandas as pa

## **Basic DataFrame Schema** <a id="basic-schema"></a>

Let's start with a simple example of defining and using a DataFrame schema:

In [4]:
# Create sample data
sample_data = pd.DataFrame(
    {
        "user_id": [1, 2, 3, 4, 5],
        "age": [25, 30, 35, 28, 45],
        "income": [50000.0, 75000.0, 90000.0, 60000.0, 120000.0],
        "category": ["A", "B", "A", "C", "B"],
    }
)

In [5]:
# Define a basic schema
basic_schema = pa.DataFrameSchema(
    {
        "user_id": pa.Column(int, pa.Check.ge(1)),  # Integer >= 1
        "age": pa.Column(int, pa.Check.in_range(18, 100)),  # Age between 18-100
        "income": pa.Column(float, pa.Check.gt(0)),  # Positive income
        "category": pa.Column(
            str, pa.Check.isin(["A", "B", "C"])
        ),  # Must be A, B, or C
    }
)

# Validate the data
try:
    validated_data = basic_schema.validate(sample_data)
    print("✅ Data validation successful!")
    print(validated_data.head())
except pa.errors.SchemaError as e:
    print(f"❌ Validation failed: {e}")

✅ Data validation successful!
   user_id  age    income category
0        1   25   50000.0        A
1        2   30   75000.0        B
2        3   35   90000.0        A
3        4   28   60000.0        C
4        5   45  120000.0        B


## **Column Validation** <a id="column-validation"></a>

Pandera provides extensive column validation capabilities:

In [6]:
# Advanced column validation examples
advanced_schema = pa.DataFrameSchema(
    {
        "user_id": pa.Column(
            int,
            checks=[
                pa.Check.ge(1),  # Greater than or equal to 1
                pa.Check(lambda s: s.is_unique, error="user_id must be unique"),
            ],
            nullable=False,  # Cannot be null
            required=True,  # Must be present
        ),
        "email": pa.Column(
            str,
            checks=[
                pa.Check(lambda s: s.str.contains("@"), error="Must contain @"),
                pa.Check(lambda s: s.str.len() >= 5, error="Email too short"),
            ],
            nullable=False,
        ),
        "score": pa.Column(
            float,
            checks=[
                pa.Check.in_range(0, 100),  # Score between 0-100
                pa.Check(
                    lambda s: (s % 0.5) == 0, error="Score must be in 0.5 increments"
                ),
            ],
            nullable=True,  # Can be null
        ),
        "optional_field": pa.Column(str, required=False),  # Optional column
    }
)

# Create test data
test_data = pd.DataFrame(
    {
        "user_id": [1, 2, 3],
        "email": ["user1@example.com", "user2@test.org", "user3@domain.net"],
        "score": [85.5, 92.0, np.nan],  # One null value
    }
)

try:
    validated = advanced_schema.validate(test_data)
    print("✅ Advanced validation successful!")
    print(validated)
except pa.errors.SchemaError as e:
    print(f"❌ Validation failed: {e}")

✅ Advanced validation successful!
   user_id              email  score
0        1  user1@example.com   85.5
1        2     user2@test.org   92.0
2        3   user3@domain.net    NaN


## **Data Type Coercion** <a id="type-coercion"></a>

Pandera can automatically coerce data types during validation:

In [7]:
# Schema with type coercion
coercion_schema = pa.DataFrameSchema(
    {
        "id": pa.Column(int, coerce=True),  # Convert to int
        "amount": pa.Column(float, coerce=True),  # Convert to float
        "active": pa.Column(bool, coerce=True),  # Convert to boolean
        "date_str": pa.Column(str, coerce=True),  # Convert to string
    }
)

# Data with mixed types
mixed_data = pd.DataFrame(
    {
        "id": ["1", "2", "3"],  # String numbers
        "amount": ["100.50", "200.75", "300.25"],  # String floats
        "active": [1, 0, 1],  # Integer booleans
        "date_str": [20230101, 20230102, 20230103],  # Integer dates
    }
)

print("Original data types:")
print(mixed_data.dtypes)
print("\nOriginal data:")
print(mixed_data)

# Validate with coercion
coerced_data = coercion_schema.validate(mixed_data)
print("\n✅ After coercion:")
print(coerced_data.dtypes)
print("\nCoerced data:")
print(coerced_data)

Original data types:
id          object
amount      object
active       int64
date_str     int64
dtype: object

Original data:
  id  amount  active  date_str
0  1  100.50       1  20230101
1  2  200.75       0  20230102
2  3  300.25       1  20230103

✅ After coercion:
id            int64
amount      float64
active         bool
date_str     object
dtype: object

Coerced data:
   id  amount  active  date_str
0   1  100.50    True  20230101
1   2  200.75   False  20230102
2   3  300.25    True  20230103


## **Custom Checks and Validation** <a id="custom-checks"></a>

Create custom validation logic for complex business rules:

In [9]:
# Custom check functions


def check_email_domain(series):
    """Check if email has valid domain"""
    valid_domains = ["gmail.com", "yahoo.com", "company.com"]
    return series.str.extract(r"@(.+)$")[0].isin(valid_domains).all()


def check_age_income_relationship(df):
    """Check if income is reasonable for age"""
    # Simple rule: income should not exceed age * 3000
    return (df["income"] <= df["age"] * 3000).all()


def check_phone_format(series):
    """Check phone number format"""
    pattern = r"^\+?\d{10,15}$"
    return series.str.match(pattern).all()


# Schema with custom checks
custom_schema = pa.DataFrameSchema(
    {
        "name": pa.Column(str, pa.Check(lambda s: s.str.len() >= 2)),
        "age": pa.Column(int, pa.Check.in_range(18, 65)),
        "email": pa.Column(
            str, pa.Check(check_email_domain, error="Invalid email domain")
        ),
        "phone": pa.Column(
            str, pa.Check(check_phone_format, error="Invalid phone format")
        ),
        "income": pa.Column(float, pa.Check.gt(0)),
    },
    checks=[pa.Check(check_age_income_relationship, error="Income too high for age")],
)

# Test data
custom_data = pd.DataFrame(
    {
        "name": ["Alice", "Bob", "Charlie"],
        "age": [25, 30, 35],
        "email": ["alice@gmail.com", "bob@yahoo.com", "charlie@company.com"],
        "phone": ["+1234567890", "9876543210", "+44123456789"],
        "income": [50000.0, 75000, 90000],
    }
)

try:
    validated = custom_schema.validate(custom_data)
    print("✅ Custom validation successful!")
    print(validated)
except pa.errors.SchemaError as e:
    print(f"❌ Validation failed: {e}")

✅ Custom validation successful!
      name  age                email         phone   income
0    Alice   25      alice@gmail.com   +1234567890  50000.0
1      Bob   30        bob@yahoo.com    9876543210  75000.0
2  Charlie   35  charlie@company.com  +44123456789  90000.0


## **DataFrame Models (Class-based API)** <a id="dataframe-models"></a>

Pandera provides a pydantic-style class-based API for defining schemas:

In [15]:
from pandera.typing import DataFrame, Series

# Define a DataFrame model


class UserDataModel(pa.DataFrameModel):
    """Schema for user data validation"""

    user_id: Series[int] = pa.Field(ge=1, unique=True)
    username: Series[str] = pa.Field()
    age: Series[int] = pa.Field()
    email: Series[str] = pa.Field()
    salary: Optional[Series[float]] = pa.Field(gt=0, nullable=True)

    # Custom validation methods
    @pa.check("email")
    def validate_email(cls, series: pd.Series) -> bool:
        """Validate email format"""
        return series.str.contains(r"^[\w\.-]+@[\w\.-]+\.\w+$").all()

    @pa.check("username")
    def validate_username(cls, series: pd.Series) -> bool:
        """Username should not contain special characters"""
        return series.str.match(r"^[a-zA-Z0-9_]+$").all()

    # Model-level validation
    @pa.dataframe_check
    def check_salary_age_relationship(cls, df: pd.DataFrame) -> bool:
        """Salary should be reasonable for age"""
        valid_salary = df["salary"].notna()
        return (
            df.loc[valid_salary, "salary"] <= df.loc[valid_salary, "age"] * 2000
        ).all()


# Test the model
user_data = pd.DataFrame(
    {
        "user_id": [1, 2, 1],
        "username": ["alice_123", "bob_user", "charlie_dev"],
        "age": [25, 30, 35],
        "email": ["alice@example.com", "bob@test.org", "charlie@company.net"],
        "salary": [45000.0, 55000.0, np.nan],  # One missing salary
    }
)

try:
    validated_users = UserDataModel.validate(user_data)
    print("✅ Model validation successful!")
    print(validated_users)
    print(f"\nSchema info: {UserDataModel.to_schema()}")
except pa.errors.SchemaError as e:
    print(f"❌ Model validation failed: {e}")

❌ Model validation failed: series 'user_id' contains duplicate values:
0    1
2    1
Name: user_id, dtype: int64


## **Error Handling and Reporting** <a id="error-handling"></a>

Pandera provides detailed error reporting and lazy validation:

In [16]:
# Create data with multiple validation errors
error_data = pd.DataFrame(
    {
        "user_id": [1, 2, 2, -1],  # Duplicate and negative ID
        "age": [25, 150, 30, 15],  # Age out of range
        "email": [
            "alice@gmail.com",
            "invalid-email",
            "bob@test.org",
            "charlie@domain.net",
        ],
        "score": [85.0, 150.0, -10.0, 95.0],  # Scores out of range
        "extra_column": ["a", "b", "c", "d"],  # Unexpected column
    }
)

# Schema with strict validation
strict_schema = pa.DataFrameSchema(
    {
        "user_id": pa.Column(int, [pa.Check.ge(1), pa.Check(lambda s: s.is_unique)]),
        "age": pa.Column(int, pa.Check.in_range(18, 100)),
        "email": pa.Column(str, pa.Check(lambda s: s.str.contains("@"))),
        "score": pa.Column(float, pa.Check.in_range(0, 100)),
    },
    strict=True,
)  # Reject extra columns

print("Data with errors:")
print(error_data)

# Regular validation (stops at first error)
print("\n--- Regular Validation ---")
try:
    strict_schema.validate(error_data)
except pa.errors.SchemaError as e:
    print(f"First error found: {e}")

# Lazy validation (collects all errors)
print("\n--- Lazy Validation ---")
try:
    strict_schema.validate(error_data, lazy=True)
except pa.errors.SchemaErrors as e:
    print("All validation errors:")
    print(e.message)
    print(f"\nNumber of errors: {len(e.schema_errors)}")

    # Access individual errors
    for i, error in enumerate(e.schema_errors[:3]):  # Show first 3 errors
        print(f"Error {i+1}: {error}")

Data with errors:
   user_id  age               email  score extra_column
0        1   25     alice@gmail.com   85.0            a
1        2  150       invalid-email  150.0            b
2        2   30        bob@test.org  -10.0            c
3       -1   15  charlie@domain.net   95.0            d

--- Regular Validation ---
First error found: column 'extra_column' not in DataFrameSchema {'user_id': <Schema Column(name=user_id, type=DataType(int64))>, 'age': <Schema Column(name=age, type=DataType(int64))>, 'email': <Schema Column(name=email, type=DataType(str))>, 'score': <Schema Column(name=score, type=DataType(float64))>}

--- Lazy Validation ---
All validation errors:
{'SCHEMA': {'COLUMN_NOT_IN_SCHEMA': [{'schema': None, 'column': None, 'check': 'column_in_schema', 'error': "column 'extra_column' not in DataFrameSchema {'user_id': <Schema Column(name=user_id, type=DataType(int64))>, 'age': <Schema Column(name=age, type=DataType(int64))>, 'email': <Schema Column(name=email, type=DataT

## **Advanced Features** <a id="advanced-features"></a>

Explore some of pandera's advanced capabilities:

In [17]:
# 1. Regex column matching
regex_schema = pa.DataFrameSchema(
    {
        "id": pa.Column(int),
        "feature_.+": pa.Column(
            float, checks=pa.Check.ge(0), regex=True
        ),  # Matches feature_1, feature_2, etc.
        "meta_.+": pa.Column(str, regex=True),  # Matches meta_info, meta_source, etc.
    }
)

regex_data = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "feature_1": [0.5, 1.2, 2.1],
        "feature_2": [1.0, 0.8, 1.5],
        "meta_info": ["A", "B", "C"],
        "meta_source": ["X", "Y", "Z"],
    }
)

validated_regex = regex_schema.validate(regex_data)
print("✅ Regex validation successful!")
print(validated_regex)

print("\n" + "=" * 50 + "\n")

# 2. Index validation
index_schema = pa.DataFrameSchema(
    columns={"value": pa.Column(float)},
    index=pa.Index(
        str, pa.Check(lambda idx: idx.str.startswith("item_")), name="item_id"
    ),
)

index_data = pd.DataFrame(
    {"value": [10.5, 20.3, 15.7]},
    index=pd.Index(["item_1", "item_2", "item_3"], name="item_id"),
)

validated_index = index_schema.validate(index_data)
print("✅ Index validation successful!")
print(validated_index)

print("\n" + "=" * 50 + "\n")

# 3. Schema transformations
base_schema = pa.DataFrameSchema({"col1": pa.Column(int), "col2": pa.Column(str)})

# Add new columns to schema
extended_schema = base_schema.add_columns(
    {"col3": pa.Column(float, pa.Check.gt(0)), "col4": pa.Column(bool)}
)

print("Base schema columns:", list(base_schema.columns.keys()))
print("Extended schema columns:", list(extended_schema.columns.keys()))

✅ Regex validation successful!
   id  feature_1  feature_2 meta_info meta_source
0   1        0.5        1.0         A           X
1   2        1.2        0.8         B           Y
2   3        2.1        1.5         C           Z


✅ Index validation successful!
         value
item_id       
item_1    10.5
item_2    20.3
item_3    15.7


Base schema columns: ['col1', 'col2']
Extended schema columns: ['col1', 'col2', 'col3', 'col4']


## **Best Practices** <a id="best-practices"></a>

Here are some best practices for using Pandera effectively:

In [ ]:
# Best Practice 1: Use descriptive error messages


def create_robust_schema():
    return pa.DataFrameSchema(
        {
            "id": pa.Column(
                int, pa.Check.ge(1, error="ID must be positive integer"), nullable=False
            ),
            "price": pa.Column(
                float,
                [
                    pa.Check.gt(0, error="Price must be positive"),
                    pa.Check.le(1000000, error="Price cannot exceed $1M"),
                ],
            ),
        }
    )


# Best Practice 2: Create reusable validation functions
def validate_data_pipeline(
    data: pd.DataFrame, schema: pa.DataFrameSchema, step_name: str = "data"
) -> pd.DataFrame:
    """Reusable validation function with logging"""
    try:
        validated = schema.validate(data, lazy=True)
        print(f"✅ {step_name} validation passed ({len(validated)} records)")
        return validated
    except pa.errors.SchemaErrors as e:
        print(f"❌ {step_name} validation failed with {len(e.schema_errors)} errors")
        for error in e.schema_errors[:3]:  # Show first 3 errors
            print(f"  - {error}")
        raise